In [0]:
df = spark.table("workspace.default.clean_churn_customers")

print("Row count:", df.count())
display(df.limit(10))

Row count: 10000


CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0
15574012,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1
15592531,822,France,Male,50,7,0.0,2,1,1,10062.8,0
15656148,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1
15792365,501,France,Male,44,4,142051.07,2,0,1,74940.5,0
15592389,684,France,Male,27,2,134603.88,1,1,1,71725.73,0


In [0]:
 

from pyspark.ml.feature import StringIndexer

df_encoded = df

categorical_columns = ["Geography", "Gender"]

for column in categorical_columns:
    output_column = f"{column}_Index"
    indexer = StringIndexer(inputCol=column, outputCol=output_column)
    df_encoded = indexer.fit(df_encoded).transform(df_encoded)

display(df_encoded.limit(10))

CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Index,Gender_Index
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0
15574012,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1,2.0,0.0
15592531,822,France,Male,50,7,0.0,2,1,1,10062.8,0,0.0,0.0
15656148,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1,1.0,1.0
15792365,501,France,Male,44,4,142051.07,2,0,1,74940.5,0,0.0,0.0
15592389,684,France,Male,27,2,134603.88,1,1,1,71725.73,0,0.0,0.0


In [0]:


from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

columns_to_normalize = ["Balance", "EstimatedSalary"]

assembler = VectorAssembler(
    inputCols=columns_to_normalize,
    outputCol="features_to_scale"
)

df_vector = assembler.transform(df_encoded)

scaler = MinMaxScaler(
    inputCol="features_to_scale",
    outputCol="scaled_features"
)

scaler_model = scaler.fit(df_vector)
df_scaled = scaler_model.transform(df_vector)

In [0]:


def extract_first(vector):
    return float(vector[0])

def extract_second(vector):
    return float(vector[1])

udf_first = udf(extract_first, DoubleType())
udf_second = udf(extract_second, DoubleType())

df_features = (
    df_scaled
    .withColumn("Normalized_Balance", udf_first("scaled_features"))
    .withColumn("Normalized_EstimatedSalary", udf_second("scaled_features"))
    .drop("features_to_scale", "scaled_features")
)

display(df_features.limit(10))

CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Index,Gender_Index,Normalized_Balance,Normalized_EstimatedSalary
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0,0.0,0.5067348931822989
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0,0.334031478677259,0.5627087386845443
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0,0.6363571759354565,0.5696543519906151
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0,0.0,0.4691200509648671
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0,0.5002462155052676,0.3954003607344501
15574012,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1,2.0,0.0,0.45339436422174445,0.7487971601287923
15592531,822,France,Male,50,7,0.0,2,1,1,10062.8,0,0.0,0.0,0.0,0.050260899915941966
15656148,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1,1.0,1.0,0.4585397202505608,0.5967334880481086
15792365,501,France,Male,44,4,142051.07,2,0,1,74940.5,0,0.0,0.0,0.5661703921301275,0.3746803819764787
15592389,684,France,Male,27,2,134603.88,1,1,1,71725.73,0,0.0,0.0,0.5364882610306042,0.35860499677719215


In [0]:
df_features.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.churn_features")

In [0]:


features_df = spark.table("workspace.default.churn_features")

print("Row count:", features_df.count())
print("Columns:", features_df.columns)

display(features_df.limit(10))

Row count: 10000
Columns: ['CustomerId', 'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_Index', 'Gender_Index', 'Normalized_Balance', 'Normalized_EstimatedSalary']


CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Index,Gender_Index,Normalized_Balance,Normalized_EstimatedSalary
15634602,619,France,Female,42,2,0.0,1,1,1,101348.88,1,0.0,1.0,0.0,0.5067348931822989
15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,2.0,1.0,0.334031478677259,0.5627087386845443
15619304,502,France,Female,42,8,159660.8,3,1,0,113931.57,1,0.0,1.0,0.6363571759354565,0.5696543519906151
15701354,699,France,Female,39,1,0.0,2,0,0,93826.63,0,0.0,1.0,0.0,0.4691200509648671
15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.1,0,2.0,1.0,0.5002462155052676,0.3954003607344501
15574012,645,Spain,Male,44,8,113755.78,2,1,0,149756.71,1,2.0,0.0,0.45339436422174445,0.7487971601287923
15592531,822,France,Male,50,7,0.0,2,1,1,10062.8,0,0.0,0.0,0.0,0.050260899915941966
15656148,376,Germany,Female,29,4,115046.74,4,1,0,119346.88,1,1.0,1.0,0.4585397202505608,0.5967334880481086
15792365,501,France,Male,44,4,142051.07,2,0,1,74940.5,0,0.0,0.0,0.5661703921301275,0.3746803819764787
15592389,684,France,Male,27,2,134603.88,1,1,1,71725.73,0,0.0,0.0,0.5364882610306042,0.35860499677719215
